# TrustShift-IDS — WP2 V11: Final Aggregation & Counterfactual Experiment

**Pre-specified objectives:** compare original 3-family median SSS with source-only-calibrated mean and max family aggregation; evaluate the same frozen neural classifiers and thresholds on matched 5,000-flow deployment windows; audit availability of a **provenance-verified** transferred XGBoost state sequence.

**Non-negotiable gates:** original V9 median states and outcomes must reproduce; all monitoring calibrations use the 54 source-null windows; deployment labels are loaded **only after** all state assignments are frozen; transferred-state outcomes are reported only when the exact 240-key stream alignment and provenance are verified. No model retraining or tuning to deployment outcomes. Mean/max comparisons are *exploratory alternative monitors*, not prospective claims if chosen after seeing V9 results.

**Runtime:** Google Colab, GPU recommended for neural inference; source calibration uses cached V9 indicators. Four seed permutations of the same IID test pool are **not independent datasets**. This notebook writes new V11 files and never overwrites V8–V10 artifacts.

In [ ]:
%pip -q install pandas numpy scipy tensorflow keras pyarrow scikit-learn
from google.colab import drive
from pathlib import Path
import os, json, gc, hashlib
import numpy as np, pandas as pd
if not os.path.ismount('/content/drive'): drive.mount('/content/drive')
BASE='/content/drive/MyDrive/TrustShift_IDS'
V2=f'{BASE}/V2_adaptive_reliability'
WP2=Path(V2)/'WP2_detector_specific'
assert WP2.exists()
SEEDS=[1,7,21,42]; WINDOW=5000; EPS=.01


## V11-A — Frozen input integrity and source-null alternative monitors

In [ ]:
V9MON=pd.read_csv(WP2/'WP2_V9_label_free_monitor_windows.csv')
V9OUT=pd.read_csv(WP2/'WP2_V9_window_policy_results.csv')
assert len(V9MON)==480 and len(V9OUT)==1920
assert not V9MON.duplicated(['detector','seed','window']).any()
assert not V9OUT.duplicated(['detector','seed','window','policy']).any()
assert set(V9MON.seed)==set(SEEDS) and set(V9MON.window)==set(range(60))
IND_CONF=['binary_confidence_shift','binary_entropy_shift','binary_margin_shift','multiclass_confidence_shift','multiclass_entropy_shift','multiclass_margin_shift']
IND_CLASS=['binary_prediction_js','multiclass_prediction_js']
IND_FEATURE=['feature_shift_mean','feature_shift_median','feature_shift_p90','feature_shift_max']
FAMS={'confidence':IND_CONF,'predicted_class':IND_CLASS,'feature':IND_FEATURE}

def e_one(v,c):
    if c['structural_zero']:return float(v>0)
    if c['scale']<=0:return 0.
    return max(0.,(float(v)-c['median'])/c['scale'])

def family_values(raw,norm,prefix=''):
    return np.array([np.median([e_one(raw[prefix+k],norm[k]) for k in inds])
                     for inds in FAMS.values()],dtype=float)

def agg(v,method):
    if method=='median':return float(np.median(v))
    if method=='mean':return float(np.mean(v))
    if method=='max':return float(np.max(v))
    raise ValueError(method)

methods=['median','mean','max']; configs={}; rows=[]; null_reports=[]
for det in ['MLP','FTTransformer']:
    with open(WP2/f'WP2_V9_{det}_frozen_monitor.json') as f: cfg=json.load(f)
    configs[det]=cfg
    null=pd.read_csv(WP2/f'WP2_V9_{det}_source_null_raw.csv')
    assert len(null)==54
    norm=cfg['normalization']
    fv_null=np.vstack([family_values(r,norm) for r in null.to_dict('records')])
    current=V9MON[V9MON.detector==det]
    fv_target=np.vstack([family_values(r,norm,'raw_') for r in current.to_dict('records')])
    for method in methods:
        src=np.array([agg(x,method) for x in fv_null]); bounds=np.quantile(src,[.95,.99])
        if method=='median':
            assert np.allclose(bounds,cfg['boundaries'],atol=1e-12,rtol=0),'V9 boundary mismatch'
        null_reports.append(dict(detector=det,aggregation=method,q95=bounds[0],q99=bounds[1],
                                 source_null_n=len(src)))
        target=np.array([agg(x,method) for x in fv_target]); states=np.where(target<=bounds[0],'normal',
                                                   np.where(target<=bounds[1],'elevated','severe'))
        for (seed,window),score,state in zip(current[['seed','window']].itertuples(index=False,name=None),target,states):
            rows.append(dict(detector=det,seed=int(seed),window=int(window),aggregation=method,
                             SSS=float(score),state=str(state),q95=float(bounds[0]),q99=float(bounds[1])))
MON11=pd.DataFrame(rows); assert len(MON11)==1440
med=MON11[MON11.aggregation=='median'].merge(V9MON[['detector','seed','window','SSS','state']],
  on=['detector','seed','window'],validate='one_to_one',suffixes=('_v11','_v9'))
assert np.allclose(med.SSS_v11,med.SSS_v9,rtol=0,atol=1e-9)
assert (med.state_v11==med.state_v9).all()
MON11.to_csv(WP2/'WP2_V11_source_only_aggregation_states.csv',index=False)
pd.DataFrame(null_reports).to_csv(WP2/'WP2_V11_source_null_aggregation_boundaries.csv',index=False)
print('V11-A: original V9 median state/SSS regression PASS')
display(MON11.groupby(['detector','aggregation','state']).size().unstack(fill_value=0))


## V11-B — Load frozen neural models and calibration thresholds

This uses the V9 neural preprocessing and original binary model definitions. The existing V9 calibration JSON is loaded instead of refitting thresholds. The V9 neural validation gates are rerun.

In [ ]:
# CELL N0 — setup, custom layers, model loading, reproduction gates
from google.colab import drive

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
import os, gc, json, glob, time, shutil
import numpy as np, pandas as pd, tensorflow as tf, keras
BASE = "/content/drive/MyDrive/TrustShift_IDS"
V2 = f"{BASE}/V2_adaptive_reliability"
OUT = f"{V2}/revision_R4"; os.makedirs(OUT, exist_ok=True)
WINDOW, EPS = 5000, 0.01
print("TF", tf.__version__, "| Keras", keras.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

# --- custom layers, verbatim from V1 Step 6.3D (needed to deserialise the FT-Transformer) ---
@tf.keras.utils.register_keras_serializable(package="TrustShiftIDS")
class NumericalFeatureTokenizer(tf.keras.layers.Layer):
    def __init__(self, n_features, token_dim, **kwargs):
        super().__init__(**kwargs); self.n_features = int(n_features); self.token_dim = int(token_dim)
    def build(self, input_shape):
        self.feature_weight = self.add_weight(name="feature_weight", shape=(self.n_features, self.token_dim), initializer="glorot_uniform", trainable=True)
        self.feature_bias = self.add_weight(name="feature_bias", shape=(self.n_features, self.token_dim), initializer="zeros", trainable=True)
        super().build(input_shape)
    def call(self, inputs):
        return tf.expand_dims(inputs, axis=-1) * self.feature_weight + self.feature_bias
    def get_config(self):
        c = super().get_config(); c.update({"n_features": self.n_features, "token_dim": self.token_dim}); return c

@tf.keras.utils.register_keras_serializable(package="TrustShiftIDS")
class CLSToken(tf.keras.layers.Layer):
    def __init__(self, token_dim, **kwargs):
        super().__init__(**kwargs); self.token_dim = int(token_dim)
    def build(self, input_shape):
        self.cls_token = self.add_weight(name="cls_token", shape=(1, 1, self.token_dim), initializer="zeros", trainable=True)
        super().build(input_shape)
    def call(self, inputs):
        return tf.concat([tf.tile(self.cls_token, [tf.shape(inputs)[0], 1, 1]), inputs], axis=1)
    def get_config(self):
        c = super().get_config(); c.update({"token_dim": self.token_dim}); return c

CUSTOM = {"NumericalFeatureTokenizer": NumericalFeatureTokenizer, "CLSToken": CLSToken,
          "TrustShiftIDS>NumericalFeatureTokenizer": NumericalFeatureTokenizer, "TrustShiftIDS>CLSToken": CLSToken}
MODELS = {"MLP": f"{BASE}/models/mlp/MLP_binary_seed42.keras",
          "FTTransformer": f"{BASE}/models/ft_transformer/FTTransformer_binary_seed42.keras"}
M = {}
for name, p in MODELS.items():
    M[name] = keras.models.load_model(p, compile=False, custom_objects=CUSTOM, safe_mode=False)
    print(f"[N0] loaded {name}: input {M[name].input_shape} output {M[name].output_shape}")

# --- frozen neural preprocessing (V1 Step 5.5C): clip + standardise continuous features only ---
SPEC = pd.read_csv(f"{BASE}/metadata/TrustShift_IDS_FROZEN_NEURAL_PREPROCESSING_SPEC.csv")
assert len(SPEC) == 68
FEATURES = SPEC["feature_id"].tolist()
CONT = SPEC[SPEC.feature_group == "continuous"]
assert len(CONT) == 56
CONT_IDX = np.array([FEATURES.index(f) for f in CONT.feature_id]); LO = CONT.lower_bound.to_numpy(float); HI = CONT.upper_bound.to_numpy(float)
MU = CONT["mean"].to_numpy(float); SD = CONT["std"].to_numpy(float)
def neural_transform(X):
    X = np.array(X, dtype=np.float64, copy=True)
    X[:, CONT_IDX] = (np.clip(X[:, CONT_IDX], LO, HI) - MU) / SD
    return X

def p_attack(model, Xn, bs=16384):
    out = model.predict(Xn.astype(np.float32), batch_size=bs, verbose=0)
    return np.asarray(out, dtype=np.float64).reshape(-1)

# Gate G1: transform(IID imputed) must equal stored IID neural representation
P_IMP = f"{BASE}/data/processed/CICIDS2017_68F_imputed"; P_NEU = f"{BASE}/data/processed/CICIDS2017_68F_neural"
iid_imp = pd.read_parquet(f"{P_IMP}/CICIDS2017_iid_test_68F_imputed.parquet")
iid_neu = pd.read_parquet(f"{P_NEU}/CICIDS2017_iid_test_68F_neural.parquet", columns=FEATURES)
d = float(np.max(np.abs(neural_transform(iid_imp[FEATURES].to_numpy()) - iid_neu.to_numpy(np.float64))))
print(f"[N0] G1 transform reproduction: max|diff| = {d:.3e}", "PASS" if d < 1e-9 else "FAIL"); assert d < 1e-9
raw = iid_imp["binary_label"]
y_iid = raw.to_numpy(np.int64) if np.issubdtype(raw.dtype, np.number) else raw.astype(str).str.lower().map({"benign": 0, "attack": 1, "0": 0, "1": 1}).to_numpy(np.int64)
X_iid_raw = iid_imp[FEATURES].to_numpy(np.float64); del iid_imp, iid_neu; gc.collect()

# Gate G2: reproduce the saved V1 validation predictions
val_neu = pd.read_parquet(f"{P_NEU}/CICIDS2017_validation_68F_neural.parquet", columns=FEATURES).to_numpy(np.float64)
for name in M:
    sp = f"{BASE}/results/{'mlp' if name=='MLP' else 'ft_transformer'}/{name}_binary_seed42_validation_predictions.parquet"
    saved = pd.read_parquet(sp); print(f"[N0] saved {name} validation columns: {list(saved.columns)}")
    cand = [c for c in saved.columns if any(k in c.lower() for k in ["prob", "p1", "p_attack", "score", "proba"])]
    p_now = p_attack(M[name], val_neu)
    if cand:
        dd = float(np.max(np.abs(p_now - saved[cand[0]].to_numpy(np.float64))))
        print(f"[N0] G2 {name}: column '{cand[0]}' max|diff| = {dd:.3e}", "PASS" if dd < 1e-4 else "CHECK")
    else:
        print(f"[N0] G2 {name}: no probability column recognised — paste the column list above")
del val_neu; gc.collect()
print("[N0] COMPLETE")

In [ ]:
calpath=Path(V2)/'revision_R4'/'N1_neural_calibration_and_thresholds.json'
assert calpath.exists(),f'Missing frozen V9 calibration: {calpath}'
with open(calpath) as f:CAL=json.load(f)
for det in ['MLP','FTTransformer']:
    assert det in CAL and all(k in CAL[det]['thresholds'] for k in ['fixed','elevated','severe'])
    assert abs(CAL[det]['T']-configs[det]['binary_temperature'])<1e-8
print('V11-B: V9 thresholds and temperature consistency PASS')
def logit(p):
    p=np.clip(p,1e-12,1-1e-12)
    return np.log(p)-np.log1p(-p)
def calibrated(p,T):return 1/(1+np.exp(-logit(p)/T))


## V11-C — All-threshold counterfactuals (no retraining)

Calculate frozen Normal, Elevated, and Severe threshold outcomes for **every** detector-window. The saved V9 `fixed_normal`, `static_severe`, and adaptive policy results must match exactly. This stage accesses deployment labels only to compute outcomes **after** all V11 state assignments have been frozen.

If a regression gate fails, stop and investigate before interpreting any new results.

In [ ]:
def count_window(y,p,t):
    pred=(p>=.5); r=1-np.maximum(p,1-p); accept=r<=t
    attack=(y==1); err=(pred!=y)
    return dict(n=len(y),acc_n=int(accept.sum()),acc_err=int((accept&err).sum()),
                n_attack=int(attack.sum()),deferred_attack=int((~accept&attack).sum()),
                acc_detected_attack=int((accept&attack&pred).sum()),
                acc_missed_attack=int((accept&attack&~pred).sum()),
                acc_false_alarm=int((accept&~attack&pred).sum()))

def label_binary(s):
    if pd.api.types.is_numeric_dtype(s):return s.to_numpy(np.int64)
    x=s.astype(str).str.lower().map({'benign':0,'attack':1,'0':0,'1':1})
    assert x.notna().all();return x.to_numpy(np.int64)

# X_iid_raw and y_iid are validated in the V9 model gate above.
COUNTER=[]
for seed in SEEDS:
    path=f'{V2}/robustness_evaluation/multiseed_streaming/V2_STEP_13B_SEED_{seed}_FLOAT64_STREAM_FEATURES.npz'
    with np.load(path,allow_pickle=False) as z:
        assert z['feature_order'].tolist()==FEATURES
        Xraw=np.asarray(z['X'],np.float64); idx=np.asarray(z['source_row_index'],np.int64)
    assert len(Xraw)==300000 and len(idx)==300000
    assert idx.min()>=0 and idx.max()<len(X_iid_raw)
    assert np.max(np.abs(Xraw[:100]-X_iid_raw[idx[:100]]))<1e-8
    y=y_iid[idx]
    Xneu=neural_transform(Xraw)
    for det in ['MLP','FTTransformer']:
        pb=calibrated(p_attack(M[det],Xneu),CAL[det]['T'])
        thresholds=CAL[det]['thresholds']
        for w in range(60):
            sl=slice(w*WINDOW,(w+1)*WINDOW)
            for state,key in [('normal','fixed'),('elevated','elevated'),('severe','severe')]:
                t=float(thresholds[key])
                COUNTER.append(dict(detector=det,seed=seed,window=w,state=state,threshold=t,
                                    **count_window(y[sl],pb[sl],t)))
        del pb
    del Xraw,Xneu,idx,y;gc.collect()
    print('counterfactual inference complete seed',seed,flush=True)
COUNTER=pd.DataFrame(COUNTER)
assert len(COUNTER)==1440 and not COUNTER.duplicated(['detector','seed','window','state']).any()
COUNTER.to_csv(WP2/'WP2_V11_all_threshold_window_counterfactuals.csv',index=False)
# Strict exact regression against V9 for every fixed and adaptive policy.
orig=V9OUT.copy()
orig['threshold_state']=np.where(orig.policy=='fixed_normal','normal',
    np.where(orig.policy=='static_severe','severe',orig.state))
orig=orig[orig.policy!='no_abstention']
check=orig.merge(COUNTER,left_on=['detector','seed','window','threshold_state'],
                 right_on=['detector','seed','window','state'],suffixes=('_v9','_v11'),validate='many_to_one')
assert len(check)==1440
fields=['n','acc_n','acc_err','n_attack','deferred_attack','acc_detected_attack','acc_missed_attack','acc_false_alarm']
for f in fields:assert (check[f+'_v9']==check[f+'_v11']).all(),f'V9 regression FAIL: {f}'
assert np.allclose(check.threshold_v9,check.threshold_v11,atol=1e-12,rtol=0)
print('V11-C: V9 policy outcomes exact regression PASS')


## V11-D — Operational outcomes for median, mean and max aggregation

All thresholds are frozen from V9 calibration. Aggregation alternatives use source-null q95/q99. No target labels enter monitor calibration. Report coverage and risk jointly, rather than selecting a winner using deployment outcomes.

In [ ]:
KEY=['detector','seed','window','state']
assert not COUNTER.duplicated(KEY).any(), 'Counterfactual outcomes must be unique'
assert not MON11.duplicated(['detector','seed','window','aggregation']).any(), 'Aggregation rows must be unique'
EVAL=MON11.merge(COUNTER,on=KEY,how='left',validate='many_to_one',indicator=True)
assert (EVAL['_merge']=='both').all(), 'Missing counterfactual outcomes'
assert len(EVAL)==len(MON11)==1440, 'Unexpected merge row count'
EVAL=EVAL.drop(columns=['_merge'])
print('V11-D many-to-one merge: PASS')
EVAL.to_csv(WP2/'WP2_V11_aggregation_policy_window_results.csv',index=False)
def summarize(g):
    rr=np.where(g.acc_n>0,g.acc_err/g.acc_n,np.nan)
    return pd.Series(dict(windows=len(g),coverage=g.acc_n.sum()/g.n.sum(),
        accepted_risk=g.acc_err.sum()/max(1,g.acc_n.sum()),
        violations=int(np.nansum(rr>EPS)),
        attack_deferral=g.deferred_attack.sum()/max(1,g.n_attack.sum()),
        auto_attack_recall=g.acc_detected_attack.sum()/max(1,g.n_attack.sum()),
        false_alarms=int(g.acc_false_alarm.sum()),
        bcdr=g.loc[np.isfinite(rr)&(rr<=EPS),'acc_detected_attack'].sum()/max(1,g.n_attack.sum())))
SUM=EVAL.groupby(['detector','aggregation']).apply(summarize,include_groups=False).reset_index()
SUM.to_csv(WP2/'WP2_V11_aggregation_policy_summary.csv',index=False)
display(SUM)
# Compare original median policy to V9 exactly, not merely approximately
v9summary=V9OUT[V9OUT.policy=='detector_specific_3family'].groupby('detector').apply(summarize,include_groups=False)
for det in ['MLP','FTTransformer']:
    a=SUM[(SUM.detector==det)&(SUM.aggregation=='median')].iloc[0]
    for k in ['coverage','accepted_risk','violations','attack_deferral','auto_attack_recall','false_alarms','bcdr']:
        assert abs(float(a[k])-float(v9summary.loc[det,k]))<1e-12,(det,k)
print('V11-D: median policy summary regression PASS')


## V11-E — Matched-coverage diagnostic, with explicit post-hoc designation

We compare each aggregation's risk and coverage with static Severe and original median. **No post-hoc matching is used to tune policies**. A descriptive nearest-coverage diagnostic is supplied; any later threshold selection based on deployment coverage must be labeled retrospective, not a prospective source-only policy.

In [ ]:
comparison=[]
for det in ['MLP','FTTransformer']:
    d=SUM[SUM.detector==det].set_index('aggregation')
    for method in ['mean','max']:
        comparison.append(dict(detector=det,alternative=method,
            coverage_difference_vs_median=float(d.loc[method,'coverage']-d.loc['median','coverage']),
            accepted_risk_difference_vs_median=float(d.loc[method,'accepted_risk']-d.loc['median','accepted_risk']),
            caveat='Exploratory alternative aggregation; not coverage-matched'))
comp=pd.DataFrame(comparison)
comp.to_csv(WP2/'WP2_V11_aggregation_tradeoffs.csv',index=False)
display(comp)


## V11-F — Transferred XGBoost state provenance gate

The repository's original clean-IID 82-window monitor is **not** the four-seed deployment state sequence. Only an XGBoost file with the same 240 `(seed,window)` keys, matching stream feature fingerprints/order, and documented XGBoost state-generation provenance may be accepted. This notebook searches source artifacts but does not infer alignment from filename or row count.

If the source sequence is unavailable, the comparison is marked **BLOCKED** and V11's aggregation experiment remains complete independently.

In [ ]:
import os
from pathlib import Path
candidates=[]
for base in [Path(V2),Path(BASE)/'results']:
    if not base.exists():continue
    for folder,dirs,files in os.walk(base):
        dirs[:]=[d for d in dirs if d!='WP2_detector_specific']
        for filename in files:
            if not filename.lower().endswith(('.csv','.parquet')):continue
            if not any(s in filename.lower() for s in ['state','monitor','stream','shift','seed','sss']):continue
            candidates.append(Path(folder)/filename)
rows=[]; aligned=[]
expected={(s,w) for s in SEEDS for w in range(60)}
for p in candidates:
    try:
        d=pd.read_csv(p) if p.suffix=='.csv' else pd.read_parquet(p)
        cols={c.lower():c for c in d.columns}
        seed=next((cols[c] for c in ['seed','random_seed','ordering_seed'] if c in cols),None)
        window=next((cols[c] for c in ['window','window_id','window_index'] if c in cols),None)
        state=next((c for c in d.columns if 'state' in c.lower()),None)
        valid=False
        if seed and window and state:
            subset=d[[seed,window,state]].copy()
            subset[seed]=pd.to_numeric(subset[seed],errors='coerce')
            subset[window]=pd.to_numeric(subset[window],errors='coerce')
            valid=(len(subset)==240 and not subset.duplicated([seed,window]).any() and
                   set(zip(subset[seed],subset[window]))==expected and
                   set(subset[state].astype(str).str.lower())<={'normal','elevated','severe'})
        rows.append(dict(path=str(p),rows=len(d),schema_aligned=bool(valid),provenance_verified=False))
        if valid:aligned.append(str(p))
    except Exception as exc:rows.append(dict(path=str(p),error=str(exc)[:120],schema_aligned=False,provenance_verified=False))
INV=pd.DataFrame(rows)
INV.to_csv(WP2/'WP2_V11_transferred_state_inventory.csv',index=False)
print('Searched',len(candidates),'files; key-aligned candidates:',len(aligned))
if aligned:print('REQUIRES MANUAL PROVENANCE VERIFICATION:',*aligned,sep='\n')
else:print('TRANSFERRED-XGBOOST BASELINE BLOCKED: no 240-window key-aligned file')


## V11-G — Final audit manifest and reporting instructions

**Do not claim:** superiority over transferred XGBoost (unless independently verified), guaranteed ≤1% risk under shift, or formal statistical independence across four orderings. The main result is a reproducible source-calibrated sensitivity comparison of aggregation rules, with exact V9 regression.

In [ ]:
manifest={
 'version':'WP2_V11','original_V8_XGBoost_clean_IID_regression':'PASS (from V8; not rerun)',
 'V9_median_monitor_regression':'PASS',
 'V9_median_policy_outcome_regression':'PASS',
 'source_only_aggregation_methods':['median','mean','max'],
 'n_neural_detector_windows':480,
 'n_aggregation_monitor_rows':len(MON11),
 'n_all_threshold_counterfactual_rows':len(COUNTER),
 'n_aggregation_outcome_rows':len(EVAL),
 'source_only_calibration':True,
 'deployment_labels_used_for_monitor_calibration':False,
 'transferred_XGBoost_state_key_aligned_candidates':len(aligned),
 'transferred_XGBoost_outcome_evaluated':False,
 'transferred_baseline_status':'BLOCKED pending independently verified exact stream provenance',
 'interpretation':'Mean/max aggregation exploratory; four orderings share underlying test observations',
 'files_written':sorted(p.name for p in WP2.glob('WP2_V11_*'))}
with open(WP2/'WP2_V11_manifest.json','w') as f:json.dump(manifest,f,indent=2)
print(json.dumps(manifest,indent=2))
print('V11 COMPLETE — send V11-A state counts, V11-D policy summary, and V11-F inventory result')
